<a href="https://colab.research.google.com/github/Cing2PO/PictureScrapper/blob/main/GelbooruScrapper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title 1. Setup & Mount Drive (Jalankan Sekali Saja)
from google.colab import drive
import os

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Buat Folder Utama (Induk)
# Semua hasil download akan masuk ke dalam folder ini di Drive Anda
base_folder = "/content/drive/MyDrive/" #@param {type:"string"}

if not os.path.exists(base_folder):
    os.makedirs(base_folder)
    print(f"✅ Folder induk dibuat: {base_folder}")
else:
    print(f"✅ Drive terhubung. Folder induk sudah ada: {base_folder}")

In [ ]:
# @title 2. Gelbooru Scraper (Run)
# @markdown Isi form dan tekan Play.

import requests
import time
import os
from tqdm.notebook import tqdm

# --- KONFIGURASI ---
search_tags = "" # @param {type:"string"}
nama_sub_folder = "" # @param {type:"string"}
rating_filter = "General (Aman)" # @param ["General (Aman)", "Sensitive (R-15)", "Questionable (R-18ish)", "Explicit (R-18)", "Semua (Tanpa Filter)"]
jumlah_halaman = 2 # @param {type:"slider", min:1, max:50, step:1}
limit_per_page = 42 # @param {type:"slider", min:20, max:100, step:1}

# --- LOGIKA PROGRAM ---
def get_rating_tag(selection):
    if selection == "General (Aman)": return " rating:general"
    if selection == "Sensitive (R-15)": return " rating:sensitive"
    if selection == "Questionable (R-18ish)": return " rating:questionable"
    if selection == "Explicit (R-18)": return " rating:explicit"
    return ""

def sanitize_name(name):
    clean = "".join([c for c in name if c.isalpha() or c.isdigit() or c in (' ', '_', '-')]).strip()
    return clean.replace(" ", "_")

def run_scraper():
    # Cek apakah Drive sudah dimount (Safety Check)
    base_folder = "/content/drive/MyDrive/Gelbooru_Download"
    if not os.path.exists(base_folder):
        print("❌ ERROR: Google Drive belum terhubung!")
        print("👉 Silakan jalankan 'Cell 1' terlebih dahulu.")
        return

    # Tentukan Sub-Folder
    if nama_sub_folder.strip() != "":
        target_folder = sanitize_name(nama_sub_folder)
    else:
        target_folder = sanitize_name(search_tags)

    full_save_path = os.path.join(base_folder, target_folder)

    if not os.path.exists(full_save_path):
        os.makedirs(full_save_path)

    # Setup Request
    final_tags = search_tags + get_rating_tag(rating_filter)
    print(f"📂 Folder: .../Gelbooru_Download/{target_folder}")
    print(f"🔍 Mencari: [{final_tags}]")
    print("-" * 40)

    base_url = "https://gelbooru.com/index.php"
    headers = {"User-Agent": "ColabScraper_V2/1.0"}
    total_downloaded = 0

    # Loop Halaman
    for page_num in range(jumlah_halaman):
        params = {
            "page": "dapi", "s": "post", "q": "index", "json": 1,
            "tags": final_tags, "limit": limit_per_page, "pid": page_num
        }

        try:
            req = requests.get(base_url, params=params, headers=headers)

            if req.status_code != 200:
                print(f"❌ Gagal akses Hal {page_num+1}")
                break

            data = req.json()
            if "post" not in data:
                print("⚠️ Tidak ada gambar lagi/Tag salah.")
                break

            posts = data["post"]

            for post in tqdm(posts, desc=f"Page {page_num+1}", leave=False):
                if "file_url" in post:
                    file_url = post["file_url"]
                    post_id = post["id"]
                    file_ext = file_url.split('.')[-1]

                    filename = os.path.join(full_save_path, f"{post_id}.{file_ext}")

                    if not os.path.exists(filename):
                        try:
                            content = requests.get(file_url, headers=headers, timeout=10).content
                            with open(filename, 'wb') as f:
                                f.write(content)
                            total_downloaded += 1
                            time.sleep(0.3)
                        except:
                            pass

            time.sleep(1)

        except Exception as e:
            print(f"Error: {e}")
            break

    print("-" * 40)
    print(f"🎉 Selesai! {total_downloaded} gambar baru disimpan.")

run_scraper()